In [ ]:
# Version 29.
# Last updated: 24th September 2026.
# Contributors: Darren Connaghan.
# A Jupyter notebook designed to be opened and run in ArcGIS Pro.

In [ ]:
import sys

print(sys.executable)
print(sys.version)

In [ ]:
import arcpy

print(arcpy.GetInstallInfo()["Version"])

# MapAction Map Automation — ArcGIS Pro 3.4.2 edition (v06)

This notebook ports `map_automation_*.py` (originally written for ArcMap 10.6 / Python 2.7 / `arcpy.mapping`) to **ArcGIS Pro 3.4.2 / Python 3 / `arcpy.mp`**.

### Running this — no conda / environment setup required
Open this notebook **inside ArcGIS Pro** (Catalog pane → right‑click a folder → New → Notebook, or double‑click this `.ipynb` if it's already in your project). Pro runs notebooks in its built‑in `arcgispro-py3` Python environment, which already has `arcpy` licensed and ready. Every import this notebook uses (`arcpy`, `os`, `sys`, `time`, `shutil`) is either standard library or already in that default environment — so there is **nothing to install and nothing to activate**. Just open it and run the cells top to bottom (Cell → Run All).

### What changed from the ArcMap version (read before running)
- `arcpy.mapping.MapDocument` (`.mxd`) → `arcpy.mp.ArcGISProject` (`.aprx`). **Your map templates must now be `.aprx` files**, not `.mxd`. The `mapsconfig()` entries below still list a couple of `.mxd` filenames from the original script (MA003, MA004, MA006) — update those to your real `.aprx` template names before running.
- ArcMap's *data frame* (`DataFrame`) did two jobs at once: holding layers, and being the thing drawn on the layout. Pro splits these into a **`Map`** (the layers) and a **`MapFrame`** (its placement/camera on a **`Layout`**). Anywhere the old code zoomed or scaled a data frame, this version now grabs the matching `MapFrame` off the layout instead.
- `ExportToJPEG(mxd, ...)` is now a method on a `Layout` object, not the project.
- `raw_input()` → `input()` (Python 3). If a layer name matches more than one dataset and can't be resolved automatically, the notebook will pause and show a text box asking you to pick one — same behaviour as before.
- Layer files: symbology is loaded via `arcpy.mp.LayerFile`, extension controlled by the single `LAYER_FILE_EXT` constant in Step 1 (set to `.lyrx` since that's what you're using — change it in one place if that ever differs).
- Picture element updates (`elem.sourceImage = ...`) and text element updates (`elem.text = ...`) use the same property names in `arcpy.mp`, so the legend‑image / title / summary swapping logic ported over largely as‑is.
- I removed the old `map_automation_updateelements.py` from this port — as written it referenced `arcpy`, `legedir`, `mapdatasources`, `mxdsdir`, and `mapsdict` without ever defining or importing them in that file, so it could not have run standalone. Its intended logic already lives (and works) inside the element‑updating step of `build_maps()` below.
- The old `absroot = appdir[:-42]` trick (chopping 42 characters off the current working directory to guess the project root) is fragile and meaningless once code lives inside a notebook rather than a fixed script path. It's replaced with an explicit `ABS_ROOT` you set in the cell below.

### Before your first real run
1. Set the paths in **Step 1**.
2. Check `mapsconfig()` in **Step 2** — confirm every template filename ends in `.aprx` and actually exists in `3_Mapping/32_Map_Templates`.
3. Confirm your `.lyr`/`.lyrx` symbology files exist in `312_Layer_files` for every entry in `dataconfig()`.
4. Run against **one map code first** (see the last cell) before batch‑running everything — several pieces (scale/zoom via `MapFrame`, the export step) are the parts most worth eyeballing on a single output before a full batch.

## Step 1 — Set your paths and imports

In [ ]:
import os
import sys
import time

import shutil
import stat
import gc
import re
from shutil import copyfile
import arcpy

# --- SET THESE FOR YOUR MACHINE / PROJECT -----------------------------------
# Root folder that contains '2_Active_Data', '3_Mapping', etc.
# (this replaces the old fragile 'chop 42 characters off cwd' trick)
ABS_ROOT = r"D:\NextCloud\Projects\2026\2026-ven-001\GIS" # "C:\Users\Evangelos\Nextcloud"
#ABS_ROOT = r"C:\Users\Evangelos\Nextcloud\Projects\2026\2026-ven-001\GIS"
#ABS_ROOT = r"C:\Users\Evangelos\Nextcloud\GIS"

sep = os.path.sep

# Derived folders — same layout as the original scripts expect
DATA_DIR   = ABS_ROOT + sep + "2_Active_Data" + sep
MAPS_DIR   = ABS_ROOT + sep + "3_Mapping" + sep
MXDS_OUT   = MAPS_DIR + "33_Map_Projects" + sep
JPGS_OUT   = MAPS_DIR + "34_Map_Products_MapAction" + sep
SYMB_DIR   = MAPS_DIR + "31_Resources" + sep + "312_Layer_files" + sep + "3121_arcpro" + sep
LEGE_DIR   = MAPS_DIR + "31_Resources" + sep + "317_Legends" + sep
TEMPLATES_DIR = MAPS_DIR + "32_Map_Templates" + sep + "321_arcpro" + sep

# Extension used for your symbology files in 312_Layer_files -- set once here, used everywhere below.
LAYER_FILE_EXT = ".lyrx"

print("ABS_ROOT set to:\n", ABS_ROOT)
print("MAPS_DIR:\n", MAPS_DIR)
print("TEMPLATES_DIR:\n", TEMPLATES_DIR)

## Step 2 — Report, map and data configuration
Ported unchanged in structure from `map_automation_configsettings.py` — these are plain Python dictionaries, nothing ArcMap-specific here. **Edit the values for your actual response / country / templates.**

⚠️ **Important:** entries in `mapsconfig()`'s `'main'` lists must **not** include a file extension — the notebook appends `LAYER_FILE_EXT` (Step 1) automatically when it loads the layer. If you paste in a `'main'` list with `.lyr`/`.lyrx` already on the names, the layer-matching check in `build_maps()` will silently find no match for every one of them — the run finishes with no error, but nothing gets added to the map.

⚠️ Every entry in a `'main'` list must **exactly** match a `'lyrs'` key in `dataconfig()` for the same map frame (`fram`), and `framconfig()`'s `'data'` values must exactly match a `'lyrs'` key too. Run the validator cell right after this one — it prints out any `'main'`/`'data'` entries that don't have a match, which is the single most common cause of "it ran but nothing was produced."

In [ ]:
def respconfig():
    respdict = [
        {'pais': ["Venezuela"],
         'iso3': ["ven"],
         'mund': ["wrl"],
         'glid': ["EQ-2026-123456-VEN"],
         'supp': ["ine"],
         'zone': ["UTC -5"],
         'size': ["a3"],  # a3, tabloid or a0
         'lang': ["ES"],  # ADM0/ADM1 label language -- must match a field suffix on the admin data,
                           # e.g. "EN" -> ADM0_EN/ADM1_EN, "ES" -> ADM0_ES/ADM1_ES. Set per response/country
                           # (maps may be produced in English, Spanish, or another language depending on
                           # the country and audience). Checked by check_admin_label_fields() in Step 2.
         'prod': ["Produced by MapAction\nmapaction.org\nvenezuela@mapaction.org"]}
    ]
    return respdict

def mapsconfig():
    mapsdict = [
        {'code': ["MA901"],
         'maps': ["country-overview-with-admin1-and-boundaries.aprx"],
         'titl': ["Country Overview with Admin 1 Boundaries and Topography"],
         'summ': ["Country overview with topography displayed"],
         'temp': ["reference"],
         # NOTE: 'main' is a single flat list shared by every map frame -- matching to the right
         # frame happens via dataconfig()'s 'fram' key, not by where a name sits in this list.
         # So each layer key should appear here ONCE ONLY, even if dataconfig() has both a
         # "Main map" and a "Location map" entry for it -- both will still be picked up correctly.
         'main': ["stle_stl_pt_natearth_cap-city",
                  "tran_air_pt_ourairports_lge-med",
                  "tran_por_pt_worldports",
                  "elev_cst_ln_all",
                  "admn_ad0_ln_all_reference",
                  "admn_ad1_ln_all_reference",
                  "phys_riv_ln_natearth",
                  "phys_lak_py_all",
                  "admn_adx_ln_all_reference",
                  "carto_fea_py_all",
                  "elev_dem_ras_all",
                  "elev_hsh_ras_all",
                  "admn_ad1_py_all_reference",
                  "admn_ad0_py_all_thematic",
                  "admn_adx_py_all_reference",
                  # admn_ad0_py_all_reference: NOT drawn for its own sake -- it's loaded here purely so
                  # framconfig()'s "Location map" entry can find it and use its full extent to set the
                  # Location map's scale (see build_maps() Step 5).
                  "admn_ad1_ln_all_location",
                  "admn_adx_ln_all_location",
                  "admn_ad0_py_all_location",
                  #"locationmap_admn_ad0_py_scaling",
                  "admn_adx_py_all_location"]},

#        {'code':["MA002"],
#         'maps':["country-overview-with-admin1-boundaries.aprx"],
#         'titl':["Country Overview with Admin 1 Boundaries"],
#         'summ':["Country overview with detail of Admin 1 boundaries and major cities"],
#         'temp':["reference"],
#         'main':["stle_stl_pt_natearth_cap-city",
#                 "tran_air_pt_ourairports_lge-med",
#                 "tran_por_pt_worldports",
#                 "elev_cst_ln_all",
#                 "admn_ad0_ln_all_reference",
#                 "admn_ad1_ln_all_reference",
#                 "phys_riv_ln_natearth",
#                 "phys_lak_py_all",
#                 "admn_adx_ln_all_reference",
#                 "carto_fea_py_all",
#                 "admn_ad1_py_all_reference",
#                 "admn_ad0_py_all_thematic",
#                 "admn_adx_py_all_reference",
#                 # admn_ad0_py_all_reference: NOT drawn for its own sake -- it's loaded here purely so
#                 # framconfig()'s "Location map" entry can find it and use its full extent to set the
#                 # Location map's scale (see build_maps() Step 5).
#                 "admn_ad1_ln_all_location",
#                 "admn_adx_ln_all_location",
#                 "admn_ad0_py_all_location",
#                 #"locationmap_admn_ad0_py_scaling",
#                 "admn_adx_py_all_location"]},

#        {'code':["MA003"],
#         'maps':["country-overview-with-admin2-boundaries-and-cities.mxd"],
#         'titl':["Country Overview with Admin 2 Boundaries"],
#         'summ':["Country overview with detail of Admin 2 boundaries and major cities"],
#         'temp':["reference"],
#         'main':["stle_stl_pt_natearth_cap-city",
#                 "tran_air_pt_ourairports_lge-med",
#                 "tran_por_pt_worldports",
#                 "elev_cst_ln_all",
#                 "admn_ad0_ln_all_reference",
#                 "admn_ad1_ln_all_reference",
#                 "admn_ad2_ln_all_reference",
#                 "phys_riv_ln_natearth",
#                 "phys_lak_py_all",
#                 "admn_adx_ln_all_reference",
#                 "carto_fea_py_all",
#                 "admn_ad2_py_all_reference",
#                 "admn_ad1_py_all_reference",
#                 "admn_ad0_py_all_thematic",
#                 "admn_adx_py_all_reference",
#                 # admn_ad0_py_all_reference: NOT drawn for its own sake -- it's loaded here purely so
#                 # framconfig()'s "Location map" entry can find it and use its full extent to set the
#                 # Location map's scale (see build_maps() Step 5).
#                 "admn_ad1_ln_all_location",
#                 "admn_adx_ln_all_location",
#                 "admn_ad0_py_all_location",
#                 #"locationmap_admn_ad0_py_scaling",
#                 "admn_adx_py_all_location"]},

#        {'code':["MA004"],
#         'maps':["country-overview-with-transport.mxd"],
#         'titl':["Country Overview with Transport"],
#         'summ':["Shows country overview with transport information such as airports and ports"],
#         'temp':["reference"],
#         'main':["stle_stl_pt_natearth_cap-city",
#                 "tran_air_pt_ourairports_lge-med",
#                 "tran_por_pt_worldports",
#                 "elev_cst_ln_all",
#                 "tran_rds_ln_natearth",
#                 "admn_ad0_ln_all_reference",
#                 "admn_ad1_ln_all_reference",
#                 "admn_ad2_ln_all_reference",
#                 "phys_riv_ln_natearth",
#                 "phys_lak_py_all",
#                 "admn_adx_ln_all_reference",
#                 "carto_fea_py_all",
#                 "admn_ad2_py_all_reference",
#                 "admn_ad1_py_all_reference",
#                 "admn_ad0_py_all_thematic",
#                 "admn_adx_py_all_reference",
#                 # admn_ad0_py_all_reference: NOT drawn for its own sake -- it's loaded here purely so
#                 # framconfig()'s "Location map" entry can find it and use its full extent to set the
#                 # Location map's scale (see build_maps() Step 5).
#                 "admn_ad1_ln_all_location",
#                 "admn_adx_ln_all_location",
#                 "admn_ad0_py_all_location",
#                 #"locationmap_admn_ad0_py_scaling",
#                 "admn_adx_py_all_location"]},
    ]
    return mapsdict

def framconfig():
    return [
        {'fram': ["Main map"], 'data': ["admn_ad0_py_all_thematic"]},
        {'fram': ["Location map"], 'data': ["admn_ad0_py_all_location"]},
    ]

def dataconfig():
    datadict = [
        {'lyrs': ["admn_ad0_py_all_thematic"], 'name': ["_admn_ad0_py_*_ine_pp*.shp"], 'iso3': [""], 'dirs': ["202_admn"], 'fram': ["Main map"]},
        {'lyrs': ["stle_stl_pt_natearth_cap-city"], 'name': ["_stle_stl_pt_*_nat*earth_pp*.shp"], 'iso3': [""], 'dirs': ["229_stle"], 'fram': ["Main map"]},
        {'lyrs': ["tran_air_pt_ourairports_lge-med"], 'name': ["_tran_air_pt_*_pp*.shp"], 'iso3': [""], 'dirs': ["232_tran"], 'fram': ["Main map"]},
        {'lyrs': ["tran_por_pt_worldports"], 'name': ["_tran_por_pt_*_pp*.shp"], 'iso3': [""], 'dirs': ["232_tran"], 'fram': ["Main map"]},
        {'lyrs': ["elev_cst_ln_all"], 'name': ["_elev_cst_ln_*_pp*.shp"], 'iso3': [""], 'dirs': ["211_elev"], 'fram': ["Main map"]},
        {'lyrs': ["admn_ad0_ln_all_reference"], 'name': ["_admn_ad0_ln_*_ine_pp*.shp"], 'iso3': [""], 'dirs': ["202_admn"], 'fram': ["Main map"]},
        {'lyrs': ["tran_rds_ln_natearth_prim-sec"], 'name': ["_tran_rds_ln_*_natearth_pp*.shp"], 'iso3': [""], 'dirs': ["232_tran"], 'fram': ["Main map"]},
        {'lyrs': ["tran_rrd_ln_all"], 'name': ["_tran_rrd_ln_*_pp*.shp"], 'iso3': [""], 'dirs': ["232_tran"], 'fram': ["Main map"]},
        {'lyrs': ["phys_riv_ln_natearth"], 'name': ["_phys_riv_ln_*_natearth_pp*.shp"], 'iso3': [""], 'dirs': ["221_phys"], 'fram': ["Main map"]},
        {'lyrs': ["phys_lak_py_all"], 'name': ["_phys_lak_py_*_pp*.shp"], 'iso3': [""], 'dirs': ["221_phys"], 'fram': ["Main map"]},
        {'lyrs': ["admn_ad1_ln_all_reference"], 'name': ["_admn_ad1_ln_*_ine_pp*.shp"], 'iso3': [""], 'dirs': ["202_admn"], 'fram': ["Main map"]},
        {'lyrs': ["admn_ad2_ln_all_reference"], 'name': ["_admn_ad2_ln_*_ine_pp*.shp"], 'iso3': [""], 'dirs': ["202_admn"], 'fram': ["Main map"]},
        {'lyrs': ["elev_dem_ras_all"], 'name': ["_elev_dem_ras_*_pp*.tif"], 'iso3': [""], 'dirs': ["211_elev"], 'fram': ["Main map"]},
        {'lyrs': ["elev_hsh_ras_all"], 'name': ["_elev_hsh_ras_*_pp*.tif"], 'iso3': [""], 'dirs': ["211_elev"], 'fram': ["Main map"]},
        {'lyrs': ["admn_ad1_py_all_reference"], 'name': ["_admn_ad1_py_*_ine_pp*.shp"], 'iso3': [""], 'dirs': ["202_admn"], 'fram': ["Main map"]},
        {'lyrs': ["admn_ad2_py_all_reference"], 'name': ["_admn_ad2_py_*_ine_pp*.shp"], 'iso3': [""], 'dirs': ["202_admn"], 'fram': ["Main map"]},
        #{'lyrs': ["admn_ad1_py_s1_ddpadmin1"], 'name': ["_admn_ad1_py_*_pp*.shp"], 'iso3': [""], 'dirs': ["202_admn"], 'fram': ["Main map"]},
        {'lyrs': ["admn_adx_ln_all_reference"], 'name': ["_admn_adx_ln_*_ine_pp*.shp"], 'iso3': ["wrl"], 'dirs': ["202_admn"], 'fram': ["Main map"]},
        {'lyrs': ["carto_fea_py_all"], 'name': ["_carto_fea_py_*_pp*.shp"], 'iso3': [""], 'dirs': ["207_carto"], 'fram': ["Main map"]},
        {'lyrs': ["admn_adx_py_all_reference"], 'name': ["_admn_adx_py_*_ine_pp*.shp"], 'iso3': ["wrl"], 'dirs': ["202_admn"], 'fram': ["Main map"]},
        # Location map entries disabled for now -- re-enable once Main map is confirmed working end-to-end.
        #{'lyrs': ["carto_fea_py_all"], 'name': ["_carto_fea_py_*_pp*.shp"], 'iso3': [""], 'dirs': ["207_carto"], 'fram': ["Location map"]},
        {'lyrs': ["elev_cst_ln_all"], 'name': ["_elev_cst_ln_*_pp*.shp"], 'iso3': [""], 'dirs': ["211_elev"], 'fram': ["Location map"]},
        {'lyrs': ["admn_ad1_ln_all_location"], 'name': ["_admn_ad1_ln_*_ine_pp*.shp"], 'iso3': [""], 'dirs': ["202_admn"], 'fram': ["Location map"]},
        {'lyrs': ["admn_adx_ln_all_location"], 'name': ["_admn_adx_ln_*_ine_pp*.shp"], 'iso3': ["wrl"], 'dirs': ["202_admn"], 'fram': ["Location map"]},
        {'lyrs': ["admn_ad0_py_all_location"], 'name': ["_admn_ad0_py_*_ine_pp*.shp"], 'iso3': [""], 'dirs': ["202_admn"], 'fram': ["Location map"]},
        {'lyrs': ["admn_adx_py_all_location"], 'name': ["_admn_adx_py_*_ine_pp*.shp"], 'iso3': ["wrl"], 'dirs': ["202_admn"], 'fram': ["Location map"]},
    ]
    return datadict

def elemconfig():
    return [
        {'temp': ["portrait_a3_marginalia_inside"], 'cenx': ["141.5"], 'ceny': ["37.5"]},
        # TODO: verify/replace these placeholder cenx/ceny once testing complete
        #{'temp': ["landscape_a3_marginalia_bottome"], 'cenx': ["141.5"], 'ceny': ["37.5"]},
        #{'temp': ["landscape_a3_marginalia_insidee"], 'cenx': ["141.5"], 'ceny': ["37.5"]},
        #{'temp': ["landscape_a0_marginalia_insidee"], 'cenx': ["141.5"], 'ceny': ["37.5"]},
        #{'temp': ["landscape_a3_marginalia_leftsidee"], 'cenx': ["141.5"], 'ceny': ["37.5"]},
        #{'temp': ["portrait_a3_marginalia_bottome"], 'cenx': ["141.5"], 'ceny': ["37.5"]},
        #{'temp': ["portrait_a0_marginalia_insidee"], 'cenx': ["141.5"], 'ceny': ["37.5"]},
    ]

### Config validator — run this every time you edit Step 2
Checks that every `'main'` entry in `mapsconfig()` has a matching `'lyrs'` key in `dataconfig()`, and that `framconfig()`'s `'data'` values match too. This is what would have caught the extension mismatch (and the other naming mismatches above) *before* running the full pipeline, instead of it finishing with no error and no output.

In [ ]:
def validate_config(mapsdict, datadict, framdict):
    """Print every mapsconfig()/framconfig() entry that has no exact match in dataconfig().
    Run this after editing Step 2, before Step 6 -- it turns the 'ran fine, nothing happened' failure
    mode into a clear list of exactly which names don't line up.
    """
    problems = []
    lyrs_by_frame = {}
    for entry in datadict:
        lyrs_by_frame.setdefault(entry['fram'][0], set()).add(entry['lyrs'][0])
    all_lyrs = set().union(*lyrs_by_frame.values()) if lyrs_by_frame else set()

    for m in mapsdict:
        mapnumb = m['code'][0]
        for mainvar in m['main']:
            base = mainvar
            for ext in (".lyrx", ".lyr"):
                if base.lower().endswith(ext):
                    problems.append(mapnumb + ": '" + mainvar + "' still has '" + ext +
                                     "' in mapsconfig() -- remove the extension, LAYER_FILE_EXT adds it automatically")
                    base = base[: -len(ext)]
            if base not in all_lyrs:
                problems.append(mapnumb + ": '" + mainvar + "' has no matching 'lyrs' entry in dataconfig() for any map frame")

    for f in framdict:
        framename = f['fram'][0]
        dataval = f['data'][0]
        if dataval not in lyrs_by_frame.get(framename, set()):
            problems.append("framconfig(): '" + dataval + "' (frame '" + framename +
                             "') has no matching 'lyrs' entry in dataconfig() for that frame")

    if problems:
        print("Found", len(problems), "config mismatch(es) -- these will silently be skipped by build_maps():")
        for p in problems:
            print(" -", p)
    else:
        print("Config check passed: every mapsconfig()/framconfig() entry has a matching dataconfig() entry.")
    return problems

### File-existence check — do the configured `.lyrx` names actually exist on disk?
`validate_config()` above only checks that names agree *between* `mapsconfig()`/`dataconfig()`/`framconfig()`. It can't tell you whether a `'lyrs'` name actually has a matching `.lyrx` file sitting in `312_Layer_files` — that's the other half of why a layer can silently fail to load. Run this after Step 1 and Step 2 are both set.

In [ ]:
def check_layer_files_exist(datadict, symb_dir, layer_file_ext):
    """For every 'lyrs' name in dataconfig(), check the matching .lyrx/.lyr file actually exists in
    312_Layer_files. Prints every miss with the exact path it looked for, so a filename typo or wrong
    folder shows up immediately instead of failing silently deep inside build_maps().
    """
    print("Looking in:\n", symb_dir)
    missing = []
    seen = set()
    for entry in datadict:
        lyrs_name = entry['lyrs'][0]
        if lyrs_name in seen:
            continue
        seen.add(lyrs_name)
        expected_path = symb_dir + lyrs_name + layer_file_ext
        if not os.path.exists(expected_path):
            missing.append(expected_path)
    if missing:
        print("Found", len(missing), "missing layer file(s):")
        for m in missing:
            print(" -", m)
    else:
        print("All", len(seen), "configured layer files were found in:\n", symb_dir)
    return missing

# run it now (uses the config loaded in Step 6 if you've already run that cell, otherwise loads its own):
_check_layer_files_exist_datadict = dataconfig()
check_layer_files_exist(_check_layer_files_exist_datadict, SYMB_DIR, LAYER_FILE_EXT)
del _check_layer_files_exist_datadict

### Admin label field check -- does the data actually have labels in the configured language?
`respconfig()`'s new `'lang'` entry (Step 2) says which language the map should be produced in --
`"EN"`, `"ES"`, or whatever suffix your admin data uses. This check looks at every **ADM0** and
**Admin 1 (ADM1)** boundary layer in `dataconfig()`, finds the real feature class on disk
(same matching logic `build_maps()` uses), and confirms it actually has an `ADM0_<lang>` /
`ADM1_<lang>` field -- e.g. `ADM0_EN`/`ADM0_ES` for country-level labels, `ADM1_EN`/`ADM1_ES` for
Admin 1. Run this after Step 1 and Step 2 are both set, same as the file-existence check above.

⚠️ This only **checks** the field exists -- it doesn't yet wire `lang` into label expressions inside
`build_maps()`/`add_layer()`. If a layer's labelling should switch fields based on `lang`, that's a
follow-up change to `add_layer()` (set `add_data.listLabelClasses()[0].expression` after load).

In [ ]:
def check_admin_label_fields(respdict, datadict, data_dir):
    """For every ADM0/ADM1 boundary layer in dataconfig(), check that the matching feature class on
    disk has an ADM0_<lang> / ADM1_<lang> field, where <lang> comes from respconfig()'s 'lang' entry
    (e.g. 'EN' -> ADM0_EN/ADM1_EN, 'ES' -> ADM0_ES/ADM1_ES). Catches a country's admin data only
    having labels in one language before a text/label element tries to read a field that isn't there --
    prints every miss with the fields that WERE found, so a suffix typo or wrong lang setting is obvious.
    Only checks ADM0/ADM1 (as requested) -- ADM2 / adx layers are skipped.
    """
    resplang = str(respdict[0].get('lang', [""])[0]).strip().upper()
    if not resplang:
        print("  * respconfig() has no 'lang' key set -- add 'lang': [\"EN\"] (or \"ES\", etc.) to "
              "respconfig() in Step 2 to run this check.")
        return []

    respiso3 = str(respdict[0]['iso3'][0])
    respmund = str(respdict[0]['mund'][0])

    admin_level_pattern = re.compile(r'admn_(ad[01])_', re.IGNORECASE)

    problems = []
    seen = set()
    for entry in datadict:
        lyrs_name = entry['lyrs'][0]
        m = admin_level_pattern.search(lyrs_name)
        if not m:
            continue  # not an ADM0/ADM1 layer (e.g. ad2, adx) -- out of scope for this check
        if lyrs_name in seen:
            continue
        seen.add(lyrs_name)

        level = m.group(1).upper()               # "AD0" or "AD1"
        admfield = level.replace("AD", "ADM") + "_" + resplang   # "ADM0_EN" / "ADM1_ES"

        entry_iso3 = entry.get('iso3', [""])[0]
        if entry_iso3 == "mund":
            prefix = respmund
        elif entry_iso3 == "reg":
            prefix = "reg"
        else:
            prefix = respiso3
        fcwildcard = prefix + entry['name'][0]

        workspace = data_dir + entry['dirs'][0]
        if not os.path.exists(workspace):
            problems.append(lyrs_name + ": data folder doesn't exist -- " + workspace)
            continue

        splittype = fcwildcard.split("_")
        if len(splittype) > 3 and splittype[3] == "ras":
            continue  # rasters have no attribute fields to check

        arcpy.env.workspace = workspace
        matches = arcpy.ListFeatureClasses(fcwildcard) or []
        if not matches:
            problems.append(lyrs_name + ": no feature class matching '" + fcwildcard + "' found in " +
                             workspace + " -- can't check label fields (see check_layer_files_exist() " +
                             "/ build_maps() for the data-matching step)")
            continue

        for fc in matches:
            fields = [f.name.upper() for f in arcpy.ListFields(os.path.join(workspace, fc))]
            if admfield not in fields:
                problems.append(lyrs_name + " (" + fc + "): missing expected label field '" + admfield +
                                 "' for lang='" + resplang + "' -- fields found: " + ", ".join(fields))

    if problems:
        print("Found", len(problems), "admin label field issue(s):")
        for p in problems:
            print(" -", p)
    else:
        print("Admin label field check passed: every ADM0/ADM1 layer has a matching '" + resplang +
              "' label field.")
    return problems


# run it now (uses the config loaded in Step 6 if you've already run that cell, otherwise loads its own):
_check_admin_labels_respdict = respconfig()
_check_admin_labels_datadict = dataconfig()
check_admin_label_fields(_check_admin_labels_respdict, _check_admin_labels_datadict, DATA_DIR)
del _check_admin_labels_respdict, _check_admin_labels_datadict

## Step 3 — Template selection
**Ratio-based selection is currently disabled.** The function below (`choose_template`) is the ported `map_automation_ratio.py` logic, kept for reference / in case you want to re-enable it later — it is not called anywhere in this notebook right now.

Instead, Step 6 hardcodes a single template and layout to cut down on clutter, per your instruction:
- Template file: `arcpro_templates_automation_test.aprx`
- Layout: `portrait_a3_marginalia_inside`

⚠️ Because the layout is now fixed by name, `build_maps()` (Step 5) selects it explicitly by name instead of blindly taking `aprx.listLayouts()[0]` — this also fixes the earlier gap where a multi-layout template could have silently exported the wrong layout.

⚠️ `elemconfig()` above only had position data for `landscape_bottom` / `landscape_side` / `portrait_bottom`. I've added a `portrait_a3_marginalia_inside` entry so the title/legend/text-updating step still runs for this layout, but its `cenx`/`ceny` legend-image coordinates are a **placeholder copied from `portrait_bottom`** — check the real position in that layout and correct it.

### A note on `.pagx`
You don't need to reference `.pagx` files anywhere in this code, and `aprx.listLayouts(name)` is the right call. `.pagx` is only an **import/export format for a single Layout's design** (used e.g. to move one layout between two different `.aprx` projects via `Layout.importLayout(path_to_pagx)`), not something a project references at runtime. Since your layouts are already saved *inside* `arcpro_templates_automation_test.aprx`, `arcpy.mp.ArcGISProject(...).listLayouts("portrait_a3_marginalia_inside")` reads them directly from the `.aprx` — the `.pagx` files sitting alongside it in the folder are irrelevant to this notebook and can stay where they are.

One thing worth confirming, though, since a name that's even slightly off (extra space, different capitalisation, underscore vs hyphen) will make `aprx.listLayouts(...)` return an empty list: run the diagnostic cell below to print every layout name actually inside your template, and check `portrait_a3_marginalia_inside` appears **exactly** as spelled.

In [ ]:
# Quick diagnostic -- run this once to confirm the exact layout name inside your template.
# Does NOT touch .pagx files; it reads layouts already saved inside the .aprx itself.
_check_path = TEMPLATES_DIR + "arcpro_templates_automation_test.aprx"
_check_aprx = arcpy.mp.ArcGISProject(_check_path)
print("Layouts found in", _check_path, ":")
for _lyt in _check_aprx.listLayouts():
    print(" -", repr(_lyt.name))
del _check_aprx

In [ ]:
def choose_template(abs_root, respdict, datadict):
    """Ratio-based template selection, ported from map_automation_ratio.py.
    NOT CALLED in this notebook right now -- Step 6 uses a fixed template/layout instead. Kept here in case
    you want to re-enable ratio-based selection later.
    """
    for resplist in range(len(respdict)):
        respiso3 = str(respdict[resplist]['iso3'][0])

    origtemp = "landscape_a3"  # default / fallback, matches original behaviour

    for datalist in range(len(datadict)):
        if datadict[datalist]['lyrs'][0] == "mainmap-admn-ad0-py-s0-scaling":
            arcpy.env.workspace = DATA_DIR + datadict[datalist]['dirs'][0]
            fcwildcard = respiso3 + "_admn_ad0_py*"
            featureclasses = arcpy.ListFeatureClasses(fcwildcard) or []
            for fc in featureclasses:
                zoomed = DATA_DIR + datadict[datalist]['dirs'][0] + sep + fc
                desc = arcpy.Describe(zoomed)
                xdist = desc.extent.XMax - desc.extent.XMin
                ydist = desc.extent.YMax - desc.extent.YMin
                ratio = xdist / ydist
                if ratio >= 1.64:
                    origtemp = "landscape_a3"
                else:
                    origtemp = "landscape_a3"  # same as original — both branches identical; adjust if needed

    origmap = TEMPLATES_DIR + "arcpro_templates_" + origtemp + ".aprx"
    print("Template selected:", origmap)
    return origmap, origtemp

## Step 4 — Add layers & swap data sources (ported from `map_automation_addtomap.py`)
`arcpy.mp.LayerFile(...)` replaces `arcpy.mapping.Layer(...)`, and `Map.addLayer(...)` replaces `arcpy.mapping.AddLayer(df, layer, position)`.

⚠️ **Correction from an earlier version of this notebook:** I initially assumed `Layer.replaceDataSource(...)` carried over unchanged from `arcpy.mapping`. It didn't — **`arcpy.mp.Layer` has no `replaceDataSource()` method at all** (`'Layer' object has no attribute 'replaceDataSource'` if you hit it). The correct approach in `arcpy.mp` is `Layer.updateConnectionProperties(current, new)`: read the layer's *current* connection dictionary, build a copy with the folder and dataset name changed, and pass both in. `swap_datasource()` below now does this instead.

⚠️ **Important behavioural difference from ArcMap:** `swap_datasource()` finds the layer to repoint by *searching the map for a layer whose name matches* the config key (`map_obj.listLayers(wildcard)`). That only works if the layer's name inside the map is exactly your config key. But the `.lyrx` file may carry whatever name it had the last time someone saved it in Pro — there's no guarantee it matches your `dataconfig()`/`mapsconfig()` key. If it doesn't, `add_layer()` still successfully adds the layer (so you *see* it appear), but `swap_datasource()`'s name search comes back empty and silently repoints nothing — no error, just an unlinked layer. That's exactly the symptom of "lyrx added but data not linked."

Fix: `add_layer()` force-renames the layer to the exact config key right after loading it, *before* adding it to the map — so the later name search in `swap_datasource()` is guaranteed to find it, regardless of whatever name was baked into the `.lyrx`.

In [ ]:
def add_layer(symb_dir, map_obj, lyr_name):
    """Load a .lyr/.lyrx symbology file, force its name to the config key so it can be found reliably
    afterwards, and add it to the bottom of the given Map."""
    lyr_path = symb_dir + lyr_name + LAYER_FILE_EXT
    try:
        layer_file = arcpy.mp.LayerFile(lyr_path)
        add_data = layer_file.listLayers()[0]
    except Exception as e:
        print("Failed to load", lyr_path, "-", e)
        return
    try:
        add_data.name = lyr_name  # force a known name so swap_datasource()'s search below is reliable
    except Exception as e:
        print("  * could not rename layer to '" + lyr_name + "' -- ", e,
              "-- swap_datasource() may not find it by name below")
    map_obj.addLayer(add_data, "BOTTOM")


def swap_datasource(wildcard, data_folder, map_obj, data_to_swap):
    """Point every layer matching `wildcard` at the real dataset on disk, using arcpy.mp's
    updateConnectionProperties() -- arcpy.mp.Layer has no replaceDataSource() method, unlike arcpy.mapping.
    """
    is_raster = data_to_swap.lower().endswith(".tif")
    is_shapefile = data_to_swap.lower().endswith(".shp")
    if not (is_raster or is_shapefile):
        print("  * unrecognised data type for", data_to_swap, "- skipping")
        return

    matched = list(map_obj.listLayers(wildcard))
    if not matched:
        print("  * no layer named '" + wildcard + "' found in map '" + map_obj.name +
              "' -- nothing to link for", data_to_swap)
        return

    for lyr in matched:
        if not lyr.supports("DATASOURCE"):
            continue
        try:
            current_cp = lyr.connectionProperties
            new_cp = dict(current_cp)
            new_cp["connection_info"] = dict(current_cp.get("connection_info", {}))
            new_cp["connection_info"]["database"] = data_folder
            # shapefiles are addressed without their extension inside a folder workspace;
            # rasters (e.g. .tif) keep the extension as the dataset name.
            new_cp["dataset"] = data_to_swap if is_raster else data_to_swap[:-4]
            lyr.updateConnectionProperties(current_cp, new_cp)
        except Exception as e:
            print("  * failed to update connection properties for", data_to_swap, "-", e)

def set_admin_label_language(map_obj, wildcard, lang):
    """Point every ADM0_<xx>/ADM1_<xx> field referenced in a layer's label expression(s) at the
    language configured in respconfig()'s 'lang' entry (e.g. 'EN' -> ADM0_EN/ADM1_EN, 'ES' ->
    ADM0_ES/ADM1_ES). Works regardless of the label expression's engine -- VBScript
    '[ADM1_EN] &vbNewLine& [ADM1_PCODE]', Python '!ADM1_EN!', or Arcade '$feature.ADM1_EN' -- since
    it only rewrites the field-name token itself, not the surrounding syntax. No-op on any layer
    whose expression doesn't reference an ADM0_xx/ADM1_xx field at all (e.g. the PCODE half of the
    expression above, or unrelated layers matched by a broad wildcard).
    """
    admin_field_pattern = re.compile(r'\bADM([01])_[A-Za-z]{2}\b', re.IGNORECASE)  # e.g. the
    # template's 'admn_adx_py_all_location' layer has 'Adm0_EN' (mixed case) rather than
    # 'ADM0_EN' -- ArcGIS itself resolves label-expression field refs case-insensitively, so
    # this matches that too; the replacement below always writes the field back out as
    # "ADM<n>_<LANG>" (the case the actual shapefile fields use, per check_admin_label_fields())
    # regardless of how the old expression had it cased.
    for lyr in map_obj.listLayers(wildcard):
        # "LABELCLASSES" isn't a valid lyr.supports() property in this arcpy version (only
        # SHOWLABELS/ISFEATURELAYER etc. are) -- only feature layers can carry label classes at
        # all, and listLabelClasses() itself is the actual source of truth, so gate on
        # ISFEATURELAYER and let the try/except below catch anything else that slips through
        # (group layers, broken layers, basemap layers, etc.).
        if not lyr.supports("ISFEATURELAYER") or not lyr.isFeatureLayer:
            continue
        try:
            label_classes = lyr.listLabelClasses()
        except Exception as e:
            print("  * could not read label classes for", lyr.name, "-", e)
            continue
# Removed this print statement
#        print("  [labels] '" + lyr.name + "' in '" + map_obj.name + "':", len(label_classes),
#              "label class(es)")
        for label_class in label_classes:
            expr = label_class.expression
# Removed this print statement
#            print("    - expression before:", repr(expr))
            if not expr or not admin_field_pattern.search(expr):
                continue
            new_expr = admin_field_pattern.sub(lambda m: "ADM" + m.group(1) + "_" + lang, expr)
            if new_expr != expr:
                try:
                    label_class.expression = new_expr
# Removed this print statement
#                    print("      -> after:         ", repr(label_class.expression))
                except Exception as e:
                    print("  * could not update label expression on", lyr.name, "-", e)

def save_aprx_with_retry(aprx_obj, max_attempts=5, delay_seconds=1.5):
    """aprx.save() can throw a bare OSError if something else briefly has the .aprx file locked --
    in particular, a desktop sync client (NextCloud/OneDrive/Dropbox/etc. -- this project lives
    under a NextCloud-synced folder) noticing the file just changed and grabbing a short-lived lock
    to checksum/start uploading it. That's exactly the kind of transient race most likely to hit
    right after we reopen a file we only just saved a moment earlier. Retry with a short pause
    rather than failing the whole map generation over a lock that typically clears within a second
    or two; only re-raise if it's still failing after every attempt.
    """
    last_err = None
    for attempt in range(1, max_attempts + 1):
        try:
            aprx_obj.save()
            return
        except OSError as e:
            last_err = e
            if attempt < max_attempts:
                print("  * aprx.save() failed (attempt", attempt, "of", max_attempts,
                      ") -- likely a sync client (NextCloud/OneDrive/etc.) briefly locking the "
                      "file. Retrying in", delay_seconds, "s...")
                time.sleep(delay_seconds)
    raise last_err

## Step 5 — Build the maps (ported from `map_automation_buildmaps.py`)
This is the main orchestration step: version/output folders, adding & swapping data per map, scaling the Main map and Location map, updating title/legend/text elements, and exporting a JPEG.

**Key Pro‑specific change:** scaling and zooming used to happen on the `DataFrame` object directly (`df.zoomToSelectedFeatures()`, `df.scale`). In Pro these operations live on a **`MapFrame`** — the object that places a `Map` onto a `Layout`. So this version looks up the layout's map frame for "Main map" / "Location map" and drives the camera/extent through that instead.

In [ ]:
def build_maps(origmap, origtemp, abs_root, respdict, datadict, mapsdict, elemdict, framdict):
    for resplist in range(len(respdict)):
        resppais = str(respdict[resplist]['pais'][0])
        respiso3 = str(respdict[resplist]['iso3'][0])
        respmund = str(respdict[resplist]['mund'][0])
        respglid = str(respdict[resplist]['glid'][0])
        respsupp = str(respdict[resplist]['supp'][0])
        respzone = str(respdict[resplist]['zone'][0])
        respprod = str(respdict[resplist]['prod'][0])
        resplang = str(respdict[resplist].get('lang', [""])[0]).strip().upper()
        if not resplang:
            print("  * respconfig() has no 'lang' key set -- ADM0/ADM1 label expressions will be left "
                  "as-is (whatever language the .lyrx template shipped with). Add 'lang': [\"EN\"] (or "
                  "\"ES\", etc.) in Step 2 to control this.")

    frammain = None
    framloca = None
    for framlist in range(len(framdict)):
        if framdict[framlist]['fram'][0] == "Main map":
            frammain = str(framdict[framlist]['data'][0])
        elif framdict[framlist]['fram'][0] == "Location map":
            framloca = str(framdict[framlist]['data'][0])

    gentime = time.strftime('%H:%M')
    gendate = time.strftime('%d %b %Y')

    count = 1
    for mapstype in range(len(mapsdict)):
        versnum = "v0" + str(count)
        mapnumb = mapsdict[mapstype]['code'][0]
        datasources = []
        missingdata = []
        admin_label_targets = []  # (mapname, wildcard) pairs to fix labels on after save+reopen below
        print("Working on map:", mapnumb,"\n")

        mapdirs = MXDS_OUT + mapnumb
        outdirs = JPGS_OUT + mapnumb
        for directory in (mapdirs, outdirs):
            if not os.path.exists(directory):
                try:
                    os.makedirs(directory)
                except Exception as e:
                    print("Failed to make", directory, "-", e)

        # work out the next version number by looking at what's already been produced for THIS map
        # code specifically. os.walk() pulls in every file under mapdirs, including ArcGIS Pro's own
        # clutter (.backups/ auto-recovery copies, GpMessages, Index, .gdb bundle contents, lock files
        # etc.) -- blindly splitting any of those on "-" and assuming the 2nd token is "vNN" crashed as
        # soon as one didn't fit that shape (ValueError: invalid literal for int() with base 10: 'g').
        # Matching strictly against "<mapnumb>-vNN-" ignores everything else in the folder.
        version_pattern = re.compile(r'^' + re.escape(mapnumb) + r'-v(\d+)-')
        maxver = 0
        for root, dirs, files in os.walk(mapdirs):
            for filez in files:
                m = version_pattern.match(filez)
                if m:
                    maxver = max(maxver, int(m.group(1)))
        if maxver:
            versnum = "v" + str(maxver + 1).zfill(2)
        os.makedirs(os.path.join(outdirs, versnum), exist_ok=True)

        finalmap = MXDS_OUT + mapnumb + sep + mapnumb + "-" + versnum + "-" + mapsdict[mapstype]['maps'][0]

        # --- create the output .aprx from the template WITHOUT touching the open template project ---
        # A plain OS-level copyfile() duplicates the .aprx's internal Map/Layout GUIDs byte-for-byte.
        # If this notebook is running from inside origmap while it's also the active project open in
        # the ArcGIS Pro GUI, arcpy's shared in-process object cache can then conflate the two by GUID
        # -- so edits meant only for the new copy also show up back in the open template project.
        # arcpy.mp.Project.saveACopy() is the ArcGIS Pro-recommended way to fork a project instead --
        # it regenerates fresh internal identifiers for the new file, so the two stay fully independent.
        template_ref = arcpy.mp.ArcGISProject(origmap)
        template_ref.saveACopy(finalmap)
        del template_ref  # just drop the Python reference -- doesn't close the template if it's open in the GUI

        # saveACopy() (like the Windows CopyFile API it wraps) carries the source file's Read-only
        # attribute over to the new copy. Template .aprx files are commonly marked Read-only on disk
        # to stop accidental edits to the master template -- which is exactly why the *output* copy
        # was opening Read-only in Pro. Clear it explicitly on the new file only; origmap is untouched.
        os.chmod(finalmap, stat.S_IWRITE | stat.S_IREAD)

        mapminusext = (JPGS_OUT + mapnumb + sep + versnum + sep + mapnumb + "-" +
                       versnum + "-" + mapsdict[mapstype]['maps'][0][:-5])  # strip '.aprx'
        finaljpg = mapminusext + ".jpg"
        os.makedirs(os.path.join(outdirs, versnum), exist_ok=True)

        aprx = arcpy.mp.ArcGISProject(finalmap)
        if os.path.normcase(os.path.abspath(aprx.filePath)) == os.path.normcase(os.path.abspath(origmap)):
            raise RuntimeError(
                "aprx just opened resolved back to the template file itself (" + origmap + ") instead of "
                "the new copy (" + finalmap + "). This means edits below would land in your template, not "
                "just the output -- close the template project in the Pro GUI (or run this notebook from a "
                "different project) before continuing."
            )

        # --- add & swap data sources, per Map in the project -------------------------------
        for map_obj in aprx.listMaps():
            mapname = map_obj.name
            for mapslist in range(len(mapsdict[mapstype]['main'])):
                mapsvar = mapsdict[mapstype]['main'][mapslist]
                for datalist in range(len(datadict)):
                    if mapname != datadict[datalist]['fram'][0]:
                        continue
                    if mapsvar != datadict[datalist]['lyrs'][0]:
                        continue

                    arcpy.env.workspace = DATA_DIR + datadict[datalist]['dirs'][0]
                    if 'iso3' not in datadict[datalist]:
                        print("  * dataconfig() entry '" + datadict[datalist]['lyrs'][0] +
                              "' has no 'iso3' key at all -- add 'iso3': [\"\"] (or 'reg'/'mund') to "
                              "that entry in Step 2. Falling back to the country iso3 for now.")
                    datalist_iso3 = datadict[datalist].get('iso3', [""])[0]
                    if datalist_iso3 == "reg":
                        fcwildcard = "reg" + datadict[datalist]['name'][0]
                    elif datalist_iso3 == "mund":
                        fcwildcard = respmund + datadict[datalist]['name'][0]
                    else:
                        fcwildcard = respiso3 + datadict[datalist]['name'][0]

                    splittype = fcwildcard.split("_")
                    if splittype[3] == "ras":
                        dataclasses = arcpy.ListRasters(fcwildcard) or []
                    else:
                        dataclasses = arcpy.ListFeatureClasses(fcwildcard) or []

                    listcount = 0
                    datatoswap = ""

                    if len(dataclasses) >= 2:
                        choosedata = ""
                        for data in dataclasses:
                            splitdata = data.split("_")
                            if splitdata[5] == respsupp:
                                datatoswap = data
                            else:
                                choosedata += "  {}: {}".format(listcount + 1, data)
                                listcount += 1
                        if len(datatoswap) == 0:
                            question = " Choose from possible datasets to match\n  '" + mapsvar + "': "
                            print(choosedata)
                            check = False
                            while not check:
                                try:
                                    data_idx = int(input(question))
                                    if data_idx >= (len(dataclasses) + 1) or data_idx == 0:
                                        print(" Try again, you entered a non viable number")
                                    else:
                                        datatoswap = dataclasses[data_idx - 1]
                                        check = True
                                except ValueError:
                                    print(" Try again, you entered more than one option or text")
                            print("")
                    if len(dataclasses) == 1:
                        # dataclasses[0] is already the full matched filename (fcwildcard above
                        # already carried the 'reg' / 'mund' / iso3 prefix into the search), so
                        # just use it as-is -- don't re-prepend the iso3 tag on top of it.
                        datatoswap = dataclasses[0]

                    if len(dataclasses) == 0:
                        missingdata.append(datadict[datalist]['lyrs'][0] + "\n")
                    elif os.path.exists(SYMB_DIR + datadict[datalist]['lyrs'][0] + LAYER_FILE_EXT):
                        splitlist = datatoswap.split("_")
                        datasources.append(splitlist[5])
                        if arcpy.Exists(datatoswap):
                            add_layer(SYMB_DIR, map_obj, datadict[datalist]['lyrs'][0])
                            swap_datasource(mapsvar, DATA_DIR + datadict[datalist]['dirs'][0],
                                             map_obj, datatoswap)

                            # ADM0/ADM1 layers ship with a fixed language baked into their .lyrx label
                            # expression (e.g. "[ADM1_EN] &vbNewLine& [ADM1_PCODE]") -- repoint that at
                            # whatever language this response is configured for. Queued rather than
                            # applied here: updateConnectionProperties() above repoints the data source,
                            # but a label class's own schema cache (set when the layer was first loaded
                            # off its .lyrx) isn't guaranteed to refresh in the same session -- a field
                            # swapped in immediately after the datasource swap can get silently dropped
                            # as "unresolved". Applying this after the save+reopen below forces that
                            # cache to refresh first.
                            if resplang:
                                admin_label_targets.append((mapname, mapsvar))

                            # These layers show neighbouring/regional admin boundaries -- the response
                            # country itself is already drawn by its own boundary layer, so exclude it
                            # here via a definition query rather than baking a fixed value into the .lyrx.
                            dq_lyrs_needing_pais_exclusion = (
                                "admn_adx_py_all_location",
                                "admn_adx_ln_all_reference",
                            )
                            if datadict[datalist]['lyrs'][0] in dq_lyrs_needing_pais_exclusion:
                                dq_expr = "ROMNAM <> '" + resppais.replace("'", "''") + "'"
                                for dq_lyr in map_obj.listLayers(mapsvar):
                                    if dq_lyr.supports("DEFINITIONQUERY"):
                                        try:
                                            dq_lyr.definitionQuery = dq_expr
                                        except Exception as e:
                                            print("  * could not set definition query on", dq_lyr.name,
                                                  "-", e)
                    else:
                        print(datadict[datalist]['lyrs'][0] + LAYER_FILE_EXT + " doesn't exist")

        save_aprx_with_retry(aprx)

        if resplang and admin_label_targets:
            # Release the current aprx's file lock before reopening the SAME path again -- same
            # reasoning as the del aprx / gc.collect() pair at the end of this function: del alone
            # only drops the Python refcount, arcpy's native file handle can linger past that, and
            # a second ArcGISProject() on a still-locked finalmap is exactly what threw the
            # OSError on save() here. map_obj (from the data-source loop above) is the only other
            # live reference to this aprx -- drop it too so nothing keeps the old Project alive.
            # original
            # del aprx, map_obj
            # gc.collect()
            # Evan
            # Release every ArcPy object that may still reference this project.
            try:
                del dq_lyr
            except NameError:
                pass

            try:
                del map_obj
            except NameError:
                pass

            del aprx
            gc.collect()

            # Give ArcGIS/Windows a moment to release native file handles.
            time.sleep(1)


            # Reopen from disk: forces a full schema refresh on every layer (including label-class
            # field caches) before we touch expressions -- see the comment above where targets are
            # queued for why this matters. aprx.listMaps(mapname) below returns fresh Map objects
            # for THIS reopened project.
            aprx = arcpy.mp.ArcGISProject(finalmap)
            for target_mapname, target_wildcard in admin_label_targets:
                for map_obj in aprx.listMaps(target_mapname):
                    set_admin_label_language(map_obj, target_wildcard, resplang)
            try:
                del map_obj  # only bound if a target_mapname above actually matched a map
            except NameError:
                pass
            gc.collect()
            save_aprx_with_retry(aprx)

        missingset = set(missingdata)
        if missingset:
            datamissing = "The following lyr files have no data for map " + mapnumb + " :\n "
            for datamiss in missingset:
                datamissing += datamiss + "\n "
            print(datamissing[:-2])

        mydatasources = set(datasources)
        mapdatasources = ""
        for datasets in mydatasources:
            mapdatasources += datasets + ", "

        # --- scale/zoom Main map + Location map via their MapFrame on the layout -------------
        # origtemp now holds the exact layout name to use (e.g. 'portrait_a3_marginalia_inside') --
        # select it explicitly rather than assuming index 0, since a template can contain several layouts.
        matching_layouts = aprx.listLayouts(origtemp)
        if not matching_layouts:
            raise ValueError(
                "No layout named '" + origtemp + "' found in " + finalmap +
                " -- check the layout name in the .aprx template."
            )
        layout = matching_layouts[0]

        loca_map = aprx.listMaps("Location map")[0]
        loca_frame = layout.listElements("MAPFRAME_ELEMENT", "Location map")[0]
        for localyr in loca_map.listLayers(framloca):
            # Location map is zoomed to 2x the extent of framloca's data (framconfig() ->
            # "admn_ad0_py_all_reference") -- i.e. computed extent's scale, doubled, so the
            # Location map shows the country boundary within a wider surrounding area.
            scalemultiple = 2
            try:
                arcpy.SelectLayerByAttribute_management(localyr)
                ext = loca_frame.getLayerExtent(localyr, True)  # True = selected features only
                loca_frame.camera.setExtent(ext)
                scale = round(loca_frame.camera.scale, -3)
                loca_frame.camera.scale = scale * scalemultiple
                arcpy.SelectLayerByAttribute_management(localyr, "CLEAR_SELECTION")
            except Exception as e:
                print("  * could not zoom/scale to", localyr.name,
                      "(likely still has an unlinked/broken data source) -", e)

        main_map = aprx.listMaps("Main map")[0]
        main_frame = layout.listElements("MAPFRAME_ELEMENT", "Main map")[0]
        for mainlyr in main_map.listLayers(frammain):
            try:
                arcpy.SelectLayerByAttribute_management(mainlyr)
                ext = main_frame.getLayerExtent(mainlyr, True)
                main_frame.camera.setExtent(ext)
                main_frame.camera.scale = round(main_frame.camera.scale, -3)
                arcpy.SelectLayerByAttribute_management(mainlyr, "CLEAR_SELECTION")
            except Exception as e:
                print("  * could not zoom/scale to", mainlyr.name,
                      "(likely still has an unlinked/broken data source) -", e)

        save_aprx_with_retry(aprx)

        # --- update title / legend / text elements on the layout -----------------------------
        for elemtype in range(len(elemdict)):
            if origtemp != elemdict[elemtype]['temp'][0]:
                continue
            for elem in layout.listElements():
                if elem.name == "legend_jpg":
                    newimage = LEGE_DIR + "ma" + mapnumb[-3:] + "_" + origtemp + ".jpg"
                    elem.sourceImage = newimage
                    elem.elementPositionX = float(elemdict[elemtype]['cenx'][0])
                    elem.elementPositionY = float(elemdict[elemtype]['ceny'][0])
                elif elem.name == "map_no":
                    elem.text = "MA" + mapnumb[-3:] + " " + versnum
                elif elem.name == "time_zone":
                    elem.text = respzone
                elif elem.name == "data_sources":
                    elem.text = mapdatasources[:-1] if mapdatasources else ""
                elif elem.name == "country":
                    elem.text = resppais
                elif elem.name == "create_date_time":
                    elem.text = str(gendate) + " / " + str(gentime)
                elif elem.name == "glide_no":
                    elem.text = respglid
                elif elem.name == "map_producer":
                    elem.text = respprod
                elif elem.name == "title":
                    elem.text = str(mapsdict[mapstype]['titl'][0])
                elif elem.name == "summary":
                    elem.text = str(mapsdict[mapstype]['summ'][0])

        save_aprx_with_retry(aprx)

        # --- export --------------------------------------------------------------------------
        try:
            print("Exporting jpg map for", mapnumb, versnum)
            layout.exportToJPEG(finaljpg, resolution=200)
        except Exception as e:
            print(mapnumb, versnum, "not exported -", e)

        # NOTE: which layout tabs are open when the output is opened is NOT controllable from
        # here. closeViews()/openView() only work against arcpy.mp.ArcGISProject("CURRENT") --
        # the live project actively displayed in the Pro GUI -- and have no effect on a project
        # opened by file path, like `aprx` here (this is documented Esri behaviour, not a bug in
        # this script). The tabs open in the output are simply whatever was open in origmap the
        # last time a person saved it interactively in Pro -- trim that in the template itself if
        # fewer open tabs are wanted; it can't be done per-map-code from a headless script.

        del aprx  # release the project's file lock before moving to the next map
        gc.collect()  # force it: del alone only drops the Python refcount, and arcpy's native
                      # file handle on finalmap.aprx can otherwise linger for as long as this
                      # kernel stays alive -- which is exactly what makes the output open
                      # read-only even in a brand-new, separate ArcGIS Pro process.
        count += 1

# 1Step 6 — Run it
Run **one map code at a time first** by temporarily filtering `mapsdict` (see the commented‑out line below) before batch‑running all of them — this is the cheapest way to confirm your templates, layer files and folder structure line up before generating a full set.

In [ ]:
# --- load configuration -------------------------------------------------------
respdict = respconfig()
mapsdict = mapsconfig()
datadict = dataconfig()
framdict = framconfig()
elemdict = elemconfig()

In [ ]:
# --- catch config mismatches (extension typos, misnamed layers) before running anything -----
problems = validate_config(mapsdict, datadict, framdict)
if problems:
    raise ValueError(str(len(problems)) + " config mismatch(es) found -- see the list printed above, "
                      "fix mapsconfig()/dataconfig()/framconfig() in Step 2, then re-run from there.")

resppais = str(respdict[0]['pais'][0])
print("\n" + time.strftime('%H:%M:%S') + " Starting to generate automated maps for '" + resppais + "'")

# --- ratio-based template selection is disabled -- fixed template/layout instead: -----------
# origmap, origtemp = choose_template(ABS_ROOT, respdict, datadict)
origmap = TEMPLATES_DIR + "arcpro_templates_automation_test.aprx"
origtemp = "portrait_a3_marginalia_inside"  # this is now the exact Layout name build_maps() looks up
print("Using fixed template:", origmap)
print("Using fixed layout:", origtemp)

build_maps(origmap, origtemp, ABS_ROOT, respdict, datadict, mapsdict, elemdict, framdict)

print("\n" + time.strftime('%H:%M:%S') + " Ending generation of automated maps")